In [4]:
!pip install transformers datasets pandas torch

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/6f/8b/69e3008d78e5cee2b30183340cc425081b78afc5eff3d080daab0adda9aa/torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata
  Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (29 kB)
  Obtaining dependency information for setuptools<82 from https://files.pythonhosted.org/packages/e1/e3/c164c88b2e5ce7b24d667b9bd83589cf4f3520d97cad01534cd3c4f55fdb/setuptools-81.0.0-py3-none-any.whl.metadata
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl.metadata
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Obtaining dependency information for networkx>=2.5.1 from https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a

In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
from string import Template

/Users/fei/Projects/YouTubeCategories-4NL3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
training_data = pd.read_csv("./codabench/app/input_data/training_data.csv")
training_label = pd.read_csv("./codabench/app/input_data/training_label.csv")
combined_training = pd.concat([training_data, training_label], axis=1)
combined_training.head()
sample = combined_training.sample(n=200)

In [14]:
test_data = pd.read_csv("./codabench/app/input_data/testing_data.csv")
test_label = pd.read_csv("./codabench/app/reference_data/testing_label.csv")
combined_test= pd.concat([test_data, test_label], axis=1)
combined_test.head()
print(len(combined_test))

279


In [12]:
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

num_to_category = {
    -1: "None",
    1: "Food & Cooking",
    2: "Video Games",
    3: "Sports & Games",
    4: "Music",
    5: "Education",
    6: "Shopping",
    7: "Vehicles",
    8: "Lifestyle",
    9: "News",
    10: "Health",
    11: "Business",
    12: "Digital Media"
}

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 12357.42it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
def predict_label(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=10)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if pred_text.startswith(prompt_text):
        pred_text = pred_text[len(prompt_text)].strip()
    match = re.search(r"\b(1[0-2]|[1-9])\b", pred_text)
    if match:
        return int(match.group(1))
    else:
        return -1

In [16]:
def test_prompt(prompt, dataset):
    correct = 0
    t_1 = Template(prompt)
    for _, row in dataset.iterrows():
        title = row["video_title"]
        description = row["video_description"]
        label = row["category"]
        prompt_text = t_1.substitute(title=title, description=description)
        pred = predict_label(prompt_text)
        pred_cat = num_to_category[pred]
        if pred_cat == label:
            correct += 1
    acc = correct / len(dataset)
    return acc


In [ ]:
prompt_1 =  """You are a topic classifier. You will be given a youtube title and it's description. You must categorize it as one of the following categories: 
    1: Food & Cooking
    2: Video Games
    3: Sports & Games
    4: Music
    5: Education
    6: Shopping
    7: Vehicles
    8: Lifestyle
    9: News
    10: Health
    11: Business
    12: Digital Media
    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading both the title and description in order to determine the most probably category. If a video title and description don't align, place more emphasize on the title.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    Here is the description: $description
    """
acc_1 = test_prompt(prompt_1, sample)
print(acc_1)
    

0.315


In [ ]:
prompt_2 =  """Given a youtube title and it's description. You must categorize it as one of the following categories, with corresponding topics that fit in the category: 
    1: Food & Cooking: Recipe, Cooking Technique
    2: Video Games: Gameplay, Game Reviews, Streaming Highlights
    3: Sports & Games: Sport Games, Commentary, Board Games, Training Specific to Sport
    4: Music: Music Videos, Karaoke Videos
    5. Education: School Coursework, Math, General Knowledge Videos, Documentary
    6: Shopping: Product Reviews, Shopping Hauls
    7: Vehicles: Car Shows, Racing
    8: Lifestyle: Traveling, Day-in-the-life, Vlogs
    9: News: Politics, Breaking News, Current Events
    10: Health & Fitness: Gym, Workout Plans, Diet
    11: Business: Investing, Economy, Entrepreneurship, Finance
    12: Digital Media: Movies, Animated Shorts, Fan Fiction, Memes

    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading both the title and description in order to determine the most probably category. If a video title and description don't align, place more emphasize on the title.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    Here is the description: $description
    """


acc_2 = test_prompt(prompt_2, sample)
print(acc_2)

Token indices sequence length is longer than the specified maximum sequence length for this model (542 > 512). Running this sequence through the model will result in indexing errors


0.46


In [ ]:
prompt_3 =  """Given a youtube title. You must categorize it as one of the following categories, with corresponding topics that fit in the category: 
    1: Food & Cooking: Recipe, Cooking Technique
    2: Video Games: Gameplay, Game Reviews, Streaming Highlights
    3: Sports & Games: Sport Games, Commentary, Board Games, Training Specific to Sport
    4: Music: Music Videos, Karaoke Videos
    5. Education: School Coursework, Math, General Knowledge Videos, Documentary
    6: Shopping: Product Reviews, Shopping Hauls
    7: Vehicles: Car Shows, Racing
    8: Lifestyle: Traveling, Day-in-the-life, Vlogs
    9: News: Politics, Breaking News, Current Events
    10: Health & Fitness: Gym, Workout Plans, Diet
    11: Business: Investing, Economy, Entrepreneurship, Finance
    12: Digital Media: Movies, Animated Shorts, Fan Fiction, Memes

    The task is to answer the classification question, "What is the main category of this YouTube video?". You can achieve this by reading the tile.
    Answer only with the corresponding number of the category names.
    Here is the title: $title
    """


acc_3 = test_prompt(prompt_3, sample)
print(acc_3)

0.38


In [17]:
best_prompt = prompt_2

acc_test = test_prompt(best_prompt, combined_test)
print(acc_test)

0.4551971326164875
